# SMART Baseline Training (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-baseline/notebooks/smart-baseline_colab.ipynb)

This notebook owns the **training stage** only:
- reuse persisted processed SMART data,
- bind the canonical SMART processed-data path when needed,
- train the SMART baseline checkpoint,
- write checkpoint and training artifacts.


In [1]:
# Step 1: Sync this repo and bootstrap deterministic Colab runtime
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


[repo] existing checkout: /content/wosac-sim-agents-experiments
[drive] /content/drive already mounted
[drive] Verified read/write access: /content/drive/MyDrive/wosac_experiments
[setup] Starting deterministic environment bootstrap
[setup] $ /usr/bin/python3 -c import jax, waymax, numpy as np, pandas as pd, scipy, sklearn; from numpy._core.umath import _center, _expandtabs; print('ok', np.__version__, pd.__version__, scipy.__version__, sklearn.__version__, jax.__version__)
[setup] Core runtime already healthy; skipping dependency install (ok 2.2.6 2.2.3 1.14.1 1.6.1 0.7.2).
[setup] $ /usr/bin/python3 -c import numpy as np; from numpy._core.umath import _center, _expandtabs; print(np.__version__, np.__file__)
[setup] NumPy probe passed (2.2.6 /usr/local/lib/python3.12/dist-packages/numpy/__init__.py).
[setup] $ /usr/bin/python3 -c import jax, waymax, numpy as np, pandas as pd, scipy, sklearn; from numpy._core.umath import _center, _expandtabs; print('ok', np.__version__, pd.__version__

In [2]:
# Suggested defaults: use the persisted SMART processed dataset and train only.
%env WOSAC_RUN_MODE=full
%env SMART_BASELINE_PROFILE=paper_repro
%env SMART_PROCESSED_DATA_ROOT=/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
%env SMART_RUN_PREPROCESS=0
%env SMART_RUN_TRAIN=1


env: WOSAC_RUN_MODE=full
env: SMART_BASELINE_PROFILE=paper_repro
env: SMART_PROCESSED_DATA_ROOT=/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
env: SMART_RUN_PREPROCESS=0
env: SMART_RUN_TRAIN=1


In [ ]:
# %env WOSAC_RUN_MODE=pilot
# %env SMART_BASELINE_PROFILE=smoke
# %env SMART_PROCESSED_DATA_ROOT=
# %env SMART_RUN_PREPROCESS=0
# %env SMART_RUN_TRAIN=1


env: WOSAC_RUN_MODE=pilot
env: SMART_BASELINE_PROFILE=smoke
env: SMART_PROCESSED_DATA_ROOT=
env: SMART_RUN_PREPROCESS=0
env: SMART_RUN_TRAIN=1


In [4]:
# Step 2: Load config, resolve processed roots, and verify training data readiness
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config


def _bool_env(name: str, default: bool = False) -> bool:
    text = os.environ.get(name, '').strip().lower()
    if text in {'1', 'true', 'yes', 'y', 'on'}:
        return True
    if text in {'0', 'false', 'no', 'n', 'off'}:
        return False
    return bool(default)


def _count_processed(split_dir: Path) -> int:
    files = [p for p in split_dir.glob('*.pkl') if p.is_file()]
    files += [p for p in split_dir.glob('*.pickle') if p.is_file()]
    return len(files)


def _bind_canonical_processed_root(actual_root: Path, canonical_root: Path) -> dict:
    actual_root = actual_root.expanduser()
    canonical_root = canonical_root.expanduser()
    if actual_root.resolve() == canonical_root.resolve():
        canonical_root.mkdir(parents=True, exist_ok=True)
        return {'mode': 'direct', 'canonical_root': str(canonical_root), 'actual_root': str(actual_root)}

    canonical_root.parent.mkdir(parents=True, exist_ok=True)
    if canonical_root.is_symlink():
        if canonical_root.resolve() != actual_root.resolve():
            canonical_root.unlink()
            canonical_root.symlink_to(actual_root, target_is_directory=True)
        return {'mode': 'symlink', 'canonical_root': str(canonical_root), 'actual_root': str(actual_root)}

    if canonical_root.exists():
        has_entries = any(canonical_root.iterdir())
        if has_entries and canonical_root.resolve() != actual_root.resolve():
            raise RuntimeError(
                f'Cannot bind {canonical_root} to {actual_root}: canonical path already exists and is non-empty.'
            )
        if not has_entries:
            canonical_root.rmdir()
            canonical_root.symlink_to(actual_root, target_is_directory=True)
            return {'mode': 'symlink', 'canonical_root': str(canonical_root), 'actual_root': str(actual_root)}

    canonical_root.symlink_to(actual_root, target_is_directory=True)
    return {'mode': 'symlink', 'canonical_root': str(canonical_root), 'actual_root': str(actual_root)}


EXPERIMENT_SLUG = 'smart-baseline'
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root='.')
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root='.')
display(bundle.summary_table)

RUN = dict(cfg.get('run', {}))
SMART = dict(cfg.get('smart', {}))
SMART_PROFILES = dict(SMART.get('profiles', {}))

RUN_NAME = str(RUN.get('run_name', 'dev'))
RUN_PREFIX = str(RUN.get('run_prefix', 'smart_baseline'))
PERSIST_ROOT = str(RUN.get('persist_root', '/content/drive/MyDrive/wosac_experiments'))
RESUME_FROM_EXISTING = bool(RUN.get('resume_from_existing', True))
RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode('utf-8')).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f'{RUN_PREFIX}_{RUN_NAME}'
persist_run_dir.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT_DIR = persist_run_dir / 'outputs' / RUN_TAG
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = os.environ.get('WOSAC_RUN_MODE', 'full').strip().lower() or 'full'
if RUN_MODE not in {'pilot', 'full'}:
    print(f"Unknown WOSAC_RUN_MODE={RUN_MODE!r}; falling back to 'full'.")
    RUN_MODE = 'full'

SMART_PROFILE = os.environ.get('SMART_BASELINE_PROFILE', str(SMART.get('profile', 'paper_repro'))).strip() or 'paper_repro'
SMART_EFFECTIVE = dict(SMART)
if SMART_PROFILE in SMART_PROFILES:
    SMART_EFFECTIVE.update(dict(SMART_PROFILES[SMART_PROFILE]))

processed_root_override = os.environ.get('SMART_PROCESSED_DATA_ROOT', '').strip()
if processed_root_override:
    SMART_EFFECTIVE['processed_data_root'] = processed_root_override

BASELINE_CKPT_DIR = os.environ.get('SMART_BASELINE_CKPT_DIR', '').strip() or str(persist_run_dir / 'checkpoints' / f'smart_baseline_{SMART_PROFILE}')
SMART_RESUME_CKPT = os.environ.get('SMART_RESUME_CKPT', '').strip()
SMART_TRAIN_SEED = int(os.environ.get('SMART_TRAIN_SEED', str(SMART_EFFECTIVE.get('seed', 2))))
PROCESSED_ROOT = Path(str(SMART_EFFECTIVE.get('processed_data_root', '/content/SMART/data/waymo_processed'))).expanduser()
CANONICAL_PROCESSED_ROOT = Path('/content/SMART/data/waymo_processed')
PATH_BINDING = _bind_canonical_processed_root(PROCESSED_ROOT, CANONICAL_PROCESSED_ROOT)
PROCESSED_TRAIN_DIR = CANONICAL_PROCESSED_ROOT / 'training'
PROCESSED_VAL_DIR = CANONICAL_PROCESSED_ROOT / 'validation'
processed_train_count = _count_processed(PROCESSED_TRAIN_DIR) if PROCESSED_TRAIN_DIR.exists() else 0
processed_val_count = _count_processed(PROCESSED_VAL_DIR) if PROCESSED_VAL_DIR.exists() else 0

smoke_mode = bool(SMART_PROFILE == 'smoke' or RUN_MODE == 'pilot')
training_readiness = {
    'processed_train_count': int(processed_train_count),
    'processed_val_count': int(processed_val_count),
    'smoke_mode': bool(smoke_mode),
    'ready_for_training': bool(smoke_mode or (processed_train_count > 0 and processed_val_count > 0)),
    'reason': 'demo_profile' if smoke_mode else 'processed_splits_required',
}

print('RUN_TAG:', RUN_TAG)
print('RUN_MODE:', RUN_MODE)
print('SMART_PROFILE:', SMART_PROFILE)
print('BASELINE_CKPT_DIR:', BASELINE_CKPT_DIR)
print('RESUME_FROM_EXISTING:', RESUME_FROM_EXISTING)
print('SMART_RESUME_CKPT (optional explicit):', SMART_RESUME_CKPT or '<auto>')
print('SMART repo commit target:', SMART_EFFECTIVE.get('repo_commit', ''))
print('SMART train config:', SMART_EFFECTIVE.get('train_config', ''))
print('SMART val config:', SMART_EFFECTIVE.get('val_config', ''))
print('SMART train seed:', SMART_TRAIN_SEED)
print('SMART processed_data_root (requested):', PROCESSED_ROOT)
print('SMART processed_data_root (canonical):', CANONICAL_PROCESSED_ROOT)
print('path_binding:', json.dumps(PATH_BINDING, indent=2, sort_keys=True))
print('cfg_hash:', cfg_hash)
print('training_readiness:')
print(json.dumps(training_readiness, indent=2, sort_keys=True))


,field,value
0,slug,smart-baseline
1,title,SMART Baseline Wrapper
2,objective,Reproduce SMART baseline with thin wrapper flo...
3,notebook,experiments/smart-baseline/notebooks/smart-bas...
4,workflow,src/workflows/smart_baseline_flow.py
5,config_file,/content/wosac-sim-agents-experiments/configs/...


RUN_TAG: 20260407T112901Z
RUN_MODE: full
SMART_PROFILE: paper_repro
BASELINE_CKPT_DIR: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/checkpoints/smart_baseline_paper_repro
RESUME_FROM_EXISTING: True
SMART_RESUME_CKPT (optional explicit): <auto>
SMART repo commit target: 42e658542b03dd14e1c36c63e214994e4b8c7b36
SMART train config: experiments/smart-baseline/configs/train_scalable_paper_repro.yaml
SMART val config: experiments/smart-baseline/configs/validation_scalable_paper_repro.yaml
SMART train seed: 2
SMART processed_data_root (requested): /content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
SMART processed_data_root (canonical): /content/SMART/data/waymo_processed
path_binding: {
  "actual_root": "/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed",
  "canonical_root": "/content/SMART/data/waymo_processed",
  "mode": "direct"
}
cfg_hash: 3a55f8daa9f07df7b3389f7127e4ee88bab0f11075d1ac8e306b7777e205820f
training_readiness:
{
  "processed_train

In [5]:
# Step 3: Build SMART baseline training command plan (train-only)
from src.workflows import run_smart_baseline_flow

flow_bundle = run_smart_baseline_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root='.',
    resume_from_existing=RESUME_FROM_EXISTING,
    smart_repo_url=str(SMART_EFFECTIVE.get('repo_url', 'https://github.com/rainmaker22/SMART.git')),
    smart_repo_branch=str(SMART_EFFECTIVE.get('branch', 'main')),
    smart_repo_commit=str(SMART_EFFECTIVE.get('repo_commit', '')).strip(),
    smart_repo_dir=str(SMART_EFFECTIVE.get('repo_dir', '/content/SMART')),
    smart_train_config=str(SMART_EFFECTIVE.get('train_config', 'configs/train/train_scalable.yaml')),
    smart_val_config=str(SMART_EFFECTIVE.get('val_config', 'configs/validation/validation_scalable.yaml')),
    smart_ckpt_path=SMART_RESUME_CKPT,
    smart_save_ckpt_path=BASELINE_CKPT_DIR,
    smart_raw_data_root=str(SMART_EFFECTIVE.get('raw_data_root', '/content/SMART/data/waymo/scenario')),
    smart_processed_data_root=str(CANONICAL_PROCESSED_ROOT),
    smart_install_pyg=bool(SMART_EFFECTIVE.get('install_pyg', True)),
    smart_env_lockfile=str(SMART_EFFECTIVE.get('env_lockfile', '')).strip(),
    smart_train_seed=SMART_TRAIN_SEED,
    smart_deterministic_train=bool(SMART_EFFECTIVE.get('deterministic_train', True)),
    smart_train_launcher_path=str(SMART_EFFECTIVE.get('train_launcher_path', 'scripts/smart_train_repro.py')),
    smart_profile=SMART_PROFILE,
    smart_force_preprocess=False,
    sync_smart_repo=True,
)

print('flow_summary:')
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))
if flow_bundle.summary.get('status') == 'sync_failed':
    raise RuntimeError(
        f"SMART repo sync failed before command execution: {flow_bundle.summary.get('sync_error', 'unknown')}"
    )
print('resolved_resume_checkpoint:', flow_bundle.summary.get('smart_ckpt_path', '') or '<none>')
print('train_cmd:')
print(flow_bundle.command_plan['train_cmd'])
print('validate_cmd (not used in this notebook):')
print(flow_bundle.command_plan['validate_cmd'])


flow_summary:
{
  "checkpoint_manifest": {
    "checkpoints": [],
    "pretrain_ckpt_path": "",
    "pretrain_ckpt_sha256": "",
    "resume": {
      "candidate_sample": [],
      "checkpoint_exists": false,
      "checkpoint_path": "",
      "num_candidates": 0,
      "source": "none"
    },
    "save_ckpt_exists": false,
    "save_ckpt_path": "/content/drive/MyDrive/wosac_experiments/smart_baseline_dev/checkpoints/smart_baseline_paper_repro"
  },
  "config_hash": "ee05f38db5b3d1084cbdb71928cbb6d3d52967b17c14d4bc7b1c9bb590e2b133",
  "created_utc": "2026-04-07T11:31:31Z",
  "data_manifest": {
    "processed_data_root": "/content/SMART/data/waymo_processed",
    "raw_data_root": "/content/SMART/data/waymo/scenario",
    "splits": {
      "training": {
        "processed": {
          "exists": true,
          "listing_sha256": "0ceebaf4328f240facbc1c1b22bc878d9902316a8b12cc75b4fc053750fb2cf0",
          "num_files": 31319,
          "path": "/content/SMART/data/waymo_processed/training"

In [6]:
# Step 4: Optional training execution (preprocess is intentionally disallowed here)
import json
import subprocess

RUN_SETUP = _bool_env('SMART_RUN_SETUP', True)
RUN_PREPROCESS = _bool_env('SMART_RUN_PREPROCESS', False)
RUN_TRAIN = _bool_env('SMART_RUN_TRAIN', False)
AUTO_SETUP = _bool_env('SMART_AUTO_SETUP', True)

if RUN_PREPROCESS:
    raise RuntimeError('This is the training-only notebook. Use smart_data_prep_colab.ipynb for preprocessing.')
if RUN_TRAIN and not bool(training_readiness.get('ready_for_training', False)):
    raise RuntimeError(
        'Processed training/validation data is missing at the canonical SMART path. '
        'Persist or bind the processed dataset first, then rerun Step 2.'
    )
if RUN_TRAIN and not RUN_SETUP and AUTO_SETUP:
    RUN_SETUP = True
    print('SMART_AUTO_SETUP=1 -> enabling setup before train.')

stage_flags = {
    'run_setup': bool(RUN_SETUP),
    'run_preprocess': False,
    'run_train': bool(RUN_TRAIN),
    'auto_setup': bool(AUTO_SETUP),
    'run_mode': RUN_MODE,
}
print('stage_flags:')
print(json.dumps(stage_flags, indent=2, sort_keys=True))


def _run_cmd(cmd: str, idx: int, total: int) -> None:
    print(f'[exec {idx}/{total}] {cmd}')
    merged_cmd = f'export PYTHONUNBUFFERED=1; {cmd}'
    process = subprocess.Popen(
        ['bash', '-lc', merged_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    recent_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        recent_lines.append(line)
        if len(recent_lines) > 80:
            recent_lines.pop(0)
    return_code = int(process.wait())
    if return_code == 0:
        return
    diag = {
        'failed_command': cmd,
        'exit_code': return_code,
        'processed_train_count': int(training_readiness.get('processed_train_count', 0)),
        'processed_val_count': int(training_readiness.get('processed_val_count', 0)),
        'recent_output': ''.join(recent_lines[-20:]),
    }
    print('step4_diagnostics:')
    print(json.dumps(diag, indent=2, sort_keys=True))
    raise subprocess.CalledProcessError(return_code, ['bash', '-lc', merged_cmd])

cmds = []
if RUN_SETUP:
    cmds.append(flow_bundle.command_plan['setup_cmd'])
if RUN_TRAIN:
    cmds.append(flow_bundle.command_plan['train_cmd'])

for i, cmd in enumerate(cmds, start=1):
    _run_cmd(cmd, i, len(cmds))

print('Training execution block done.')


stage_flags:
{
  "auto_setup": true,
  "run_mode": "full",
  "run_preprocess": false,
  "run_setup": true,
  "run_train": true
}
[exec 1/2] cd /content/SMART && python -m pip install -r /content/wosac-sim-agents-experiments/experiments/smart-baseline/env/requirements-smart-exact-cu113.txt && bash scripts/install_pyg.sh
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu113
Looking in links: https://data.pyg.org/whl/torch-1.12.0+cu113.html
Ignoring torch: markers 'platform_system == "Linux" and python_version == "3.9"' don't match your environment
Ignoring torchvision: markers 'platform_system == "Linux" and python_version == "3.9"' don't match your environment
Ignoring torch-cluster: markers 'platform_system == "Linux" and python_version == "3.9"' don't match your environment
Ignoring torch-scatter: markers 'platform_system == "Linux" and python_version == "3.9"' don't match your environment
Ignoring torch-sparse: markers 'platform_system == "Linux" and pyt

CalledProcessError: Command '['bash', '-lc', 'export PYTHONUNBUFFERED=1; cd /content/SMART && python -m pip install -r /content/wosac-sim-agents-experiments/experiments/smart-baseline/env/requirements-smart-exact-cu113.txt && bash scripts/install_pyg.sh']' returned non-zero exit status 1.

In [ ]:
# Step 5: Write baseline training artifact + run manifest (Drive)
from src.workflows import build_training_run_manifest, write_json

stage_flags = globals().get('stage_flags', {'run_setup': False, 'run_preprocess': False, 'run_train': False, 'run_mode': RUN_MODE})
ckpt_files = sorted([str(p) for p in Path(BASELINE_CKPT_DIR).expanduser().glob('*.ckpt')])
resume_resolution = dict(flow_bundle.summary.get('resume_resolution', {}) or {})

payload = {
    'run_id': 'smart_baseline_train_v0',
    'status': str(flow_bundle.summary.get('status', 'unknown')),
    'run_tag': RUN_TAG,
    'run_mode': RUN_MODE,
    'smart_profile': SMART_PROFILE,
    'checkpoint_dir': BASELINE_CKPT_DIR,
    'checkpoint_files': ckpt_files,
    'resume_resolution': resume_resolution,
    'run_output_dir': str(RUN_OUTPUT_DIR),
    'stage_flags': stage_flags,
    'training_readiness': training_readiness,
    'path_binding': PATH_BINDING,
    'flow_summary': flow_bundle.summary,
    'command_plan': flow_bundle.command_plan,
    'flow_artifacts': flow_bundle.artifacts,
    'notes': [
        'Training-only notebook. Use smart_data_prep_colab.ipynb for preprocessing and smart_baseline_eval_colab.ipynb for rollout + official metrics.',
    ],
    'provenance': {
        'config_hash': cfg_hash,
        'created_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'experiment_slug': EXPERIMENT_SLUG,
    },
}

drive_path = RUN_OUTPUT_DIR / 'smart_baseline_train_v0.json'
write_json(drive_path, payload)
run_manifest = build_training_run_manifest(
    run_id='smart_baseline_train_v0',
    run_tag=RUN_TAG,
    experiment_slug=EXPERIMENT_SLUG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root='.',
    config_hash=cfg_hash,
    flow_summary=flow_bundle.summary,
    stage_flags=stage_flags,
    checkpoint_dir=BASELINE_CKPT_DIR,
    resume_checkpoint_path=str(flow_bundle.summary.get('smart_ckpt_path', '')),
    resume_checkpoint_source=str(resume_resolution.get('source', '')),
    extra={
        'flow_artifacts': flow_bundle.artifacts,
        'checkpoint_files': ckpt_files,
        'training_readiness': training_readiness,
        'path_binding': PATH_BINDING,
    },
)
manifest_path = RUN_OUTPUT_DIR / 'run_manifest.json'
write_json(manifest_path, run_manifest)

print('drive_path:', drive_path)
print('manifest_path:', manifest_path)
print('num_ckpts_found:', len(ckpt_files))


drive_path: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/smart_baseline_train_v0.json
manifest_path: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/run_manifest.json
num_ckpts_found: 5
